# Day 4: 生成模型与逆向MOF设计 - 完整教程

本教程将带你深入学习生成模型在MOF设计中的应用，包括：
- **VAE（变分自编码器）**: 学习MOF的潜在表示
- **扩散模型**: 高质量MOF生成
- **贝叶斯优化**: 性质导向的逆向设计

## 目录
1. [环境设置与依赖](#part1)
2. [MOF-VAE训练与潜在空间](#part2)
3. [扩散模型训练与采样](#part3)
4. [贝叶斯优化逆向设计](#part4)
5. [可视化分析](#part5)
6. [端到端设计流程](#part6)
7. [总结与练习](#part7)

---
<a id="part1"></a>
## Part 1: 环境设置与依赖

首先确保安装了所有必要的库。

In [ ]:
# 导入基础库
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch_geometric.data import Data, DataLoader
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# 导入我们的模型
from src.generative_models.mof_vae import MOFVAE, MOFVAETrainer
from src.generative_models.mof_diffusion import MOFDiffusion, MOFDiffusionTrainer
from src.generative_models.bayesian_optimizer import (
    GaussianProcess, BayesianOptimizer, MOFInverseDesigner
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

### 生成模拟MOF数据集

为了演示，我们创建一些模拟的MOF图数据。

In [ ]:
def generate_synthetic_mof_dataset(num_samples=100, num_nodes=30, node_dim=4, edge_dim=24):
    """
    生成模拟的MOF图数据集
    
    参数:
        num_samples: 生成的MOF数量
        num_nodes: 每个MOF的节点数
        node_dim: 节点特征维度
        edge_dim: 边特征维度
    """
    dataset = []
    
    for i in range(num_samples):
        # 随机节点特征（模拟原子属性）
        x = torch.randn(num_nodes, node_dim)
        
        # 随机边（每个节点平均连接3-5个邻居）
        num_edges = np.random.randint(num_nodes * 3, num_nodes * 5)
        edge_index = torch.randint(0, num_nodes, (2, num_edges))
        
        # 边特征（模拟距离的Gaussian扩展）
        edge_attr = torch.randn(num_edges, edge_dim)
        
        # 模拟性质（如CO2吸附量）
        # 简单起见，用节点特征的均值和方差来模拟
        co2_uptake = 5.0 + 2.0 * torch.mean(x).item() + np.random.randn() * 0.5
        
        data = Data(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=torch.tensor([co2_uptake]),
            num_nodes=num_nodes
        )
        
        dataset.append(data)
    
    return dataset

# 生成数据集
print("生成模拟MOF数据集...")
train_dataset = generate_synthetic_mof_dataset(num_samples=200, num_nodes=30)
val_dataset = generate_synthetic_mof_dataset(num_samples=50, num_nodes=30)

print(f"训练集大小: {len(train_dataset)}")
print(f"验证集大小: {len(val_dataset)}")
print(f"\n示例数据:")
print(f"  节点特征形状: {train_dataset[0].x.shape}")
print(f"  边索引形状: {train_dataset[0].edge_index.shape}")
print(f"  边特征形状: {train_dataset[0].edge_attr.shape}")
print(f"  CO2吸附量: {train_dataset[0].y.item():.2f} mmol/g")

---
<a id="part2"></a>
## Part 2: MOF-VAE训练与潜在空间探索

变分自编码器（VAE）学习MOF的低维潜在表示，使我们能够：
- 在潜在空间中进行插值
- 生成新的MOF结构
- 进行性质导向的优化

### 2.1 创建和训练VAE模型

In [ ]:
# 创建DataLoader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# 创建VAE模型
vae_model = MOFVAE(
    node_input_dim=4,
    edge_input_dim=24,
    hidden_dim=64,
    latent_dim=16,  # 潜在空间维度
    max_num_nodes=30,
    num_conv_layers=3,
    beta=0.5,  # KL散度权重
    dropout=0.1
).to(device)

print("VAE模型结构:")
print(f"  潜在空间维度: {vae_model.latent_dim}")
print(f"  隐藏层维度: {vae_model.hidden_dim}")
print(f"  卷积层数: {vae_model.num_conv_layers}")
print(f"  β (KL权重): {vae_model.beta}")
print(f"\n总参数量: {sum(p.numel() for p in vae_model.parameters()):,}")

In [ ]:
# 创建训练器
optimizer = torch.optim.Adam(vae_model.parameters(), lr=1e-3)
vae_trainer = MOFVAETrainer(vae_model, optimizer, device=device)

# 训练模型
print("开始训练VAE...")
num_epochs = 50
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # 训练
    train_loss = vae_trainer.train_epoch(train_loader)
    train_losses.append(train_loss)
    
    # 验证
    val_loss = vae_trainer.validate(val_loader)
    val_losses.append(val_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"  训练损失: {train_loss['loss']:.4f} (重构={train_loss['recon_loss']:.4f}, KL={train_loss['kl_loss']:.4f})")
        print(f"  验证损失: {val_loss['loss']:.4f}")

print("\n训练完成！")

### 2.2 可视化训练过程

In [ ]:
# 绘制损失曲线
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 总损失
axes[0].plot([d['loss'] for d in train_losses], label='Train', alpha=0.7)
axes[0].plot([d['loss'] for d in val_losses], label='Val', alpha=0.7)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Total Loss')
axes[0].set_title('Total Loss (ELBO)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 重构损失
axes[1].plot([d['recon_loss'] for d in train_losses], label='Train', alpha=0.7)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Reconstruction Loss')
axes[1].set_title('Reconstruction Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# KL散度
axes[2].plot([d['kl_loss'] for d in train_losses], label='Train', alpha=0.7)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('KL Divergence')
axes[2].set_title('KL Divergence')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.3 探索潜在空间

In [ ]:
# 将所有训练数据编码到潜在空间
vae_model.eval()
latent_vectors = []
properties = []

with torch.no_grad():
    for batch in train_loader:
        batch = batch.to(device)
        mu, logvar = vae_model.encode(batch)
        z = vae_model.reparameterize(mu, logvar)
        
        latent_vectors.append(z.cpu())
        properties.append(batch.y.cpu())

latent_vectors = torch.cat(latent_vectors, dim=0).numpy()
properties = torch.cat(properties, dim=0).numpy()

print(f"潜在向量形状: {latent_vectors.shape}")
print(f"性质范围: {properties.min():.2f} - {properties.max():.2f} mmol/g")

In [ ]:
# 使用t-SNE降维到2D可视化
print("使用t-SNE降维...")
tsne = TSNE(n_components=2, random_state=42)
latent_2d = tsne.fit_transform(latent_vectors)

# 绘制潜在空间（按性质着色）
plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    latent_2d[:, 0], 
    latent_2d[:, 1],
    c=properties,
    cmap='viridis',
    s=50,
    alpha=0.6
)
plt.colorbar(scatter, label='CO2 Uptake (mmol/g)')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.title('VAE潜在空间可视化（按CO2吸附量着色）')
plt.grid(True, alpha=0.3)
plt.show()

print("\n观察: 具有相似性质的MOF在潜在空间中聚集")

### 2.4 潜在空间插值

In [ ]:
# 选择两个不同的MOF进行插值
idx1, idx2 = 0, 50
z1 = torch.tensor(latent_vectors[idx1:idx1+1]).to(device)
z2 = torch.tensor(latent_vectors[idx2:idx2+1]).to(device)

print(f"MOF 1 CO2吸附量: {properties[idx1]:.2f} mmol/g")
print(f"MOF 2 CO2吸附量: {properties[idx2]:.2f} mmol/g")

# 生成插值点
num_interpolations = 10
alphas = np.linspace(0, 1, num_interpolations)
interpolated_mofs = []

vae_model.eval()
with torch.no_grad():
    for alpha in alphas:
        # 线性插值
        z_interp = (1 - alpha) * z1 + alpha * z2
        
        # 解码
        node_gen, adj_gen, edge_gen = vae_model.decode(z_interp, num_nodes=30)
        interpolated_mofs.append({
            'alpha': alpha,
            'node_features': node_gen.cpu(),
            'adj_matrix': adj_gen.cpu(),
            'edge_features': edge_gen.cpu()
        })

print(f"\n生成了 {len(interpolated_mofs)} 个插值MOF")
print("这些MOF代表了从MOF 1到MOF 2的平滑过渡")

### 2.5 从随机采样生成新MOF

In [ ]:
# 从标准正态分布采样生成新MOF
num_samples = 5
print(f"从潜在空间采样生成 {num_samples} 个新MOF...\n")

with torch.no_grad():
    node_gen, adj_gen, edge_gen = vae_model.sample(num_samples=num_samples)

for i in range(num_samples):
    num_edges = (adj_gen[i] > 0.5).sum().item()
    print(f"生成的MOF {i+1}:")
    print(f"  节点特征: {node_gen[i].shape}")
    print(f"  边数: {num_edges}")
    print()

---
<a id="part3"></a>
## Part 3: 扩散模型训练与采样

扩散模型通过逐步去噪过程生成高质量的MOF结构。

### 3.1 创建和训练扩散模型

In [ ]:
# 创建扩散模型
diffusion_model = MOFDiffusion(
    node_dim=4,
    edge_dim=24,
    hidden_dim=64,
    num_conv_layers=3,
    num_timesteps=100,  # 时间步数（简化版）
    schedule='cosine',  # 噪声调度策略
    dropout=0.1
).to(device)

print("扩散模型结构:")
print(f"  时间步数: {diffusion_model.num_timesteps}")
print(f"  噪声调度: {diffusion_model.schedule}")
print(f"  隐藏层维度: {diffusion_model.hidden_dim}")
print(f"\n总参数量: {sum(p.numel() for p in diffusion_model.parameters()):,}")

In [ ]:
# 训练扩散模型（注意：这可能需要较长时间）
optimizer_diff = torch.optim.Adam(diffusion_model.parameters(), lr=1e-3)
diff_trainer = MOFDiffusionTrainer(diffusion_model, optimizer_diff, device=device)

print("开始训练扩散模型...")
num_epochs_diff = 30  # 较少的epoch用于演示
diff_losses = []

for epoch in range(num_epochs_diff):
    loss = diff_trainer.train_epoch(train_loader)
    diff_losses.append(loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs_diff}: Loss = {loss:.4f}")

print("\n训练完成！")

In [ ]:
# 绘制扩散模型训练损失
plt.figure(figsize=(8, 5))
plt.plot(diff_losses, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Diffusion Loss')
plt.title('扩散模型训练损失')
plt.grid(True, alpha=0.3)
plt.show()

### 3.2 从扩散模型采样生成MOF

In [ ]:
# 需要提供图的拓扑结构（边索引和边特征）
# 这里使用训练集中的一个作为模板
template = train_dataset[0].to(device)

print("使用扩散模型生成MOF...")
print(f"模板图: {template.num_nodes} 个节点, {template.edge_index.shape[1]} 条边")

# 从纯噪声开始生成
diffusion_model.eval()
with torch.no_grad():
    generated_nodes = diffusion_model.sample(
        num_nodes=template.num_nodes,
        edge_index=template.edge_index,
        edge_attr=template.edge_attr,
        return_trajectory=False  # 不返回中间步骤
    )

print(f"\n生成的节点特征: {generated_nodes.shape}")
print("扩散模型成功从噪声生成了MOF结构！")

---
<a id="part4"></a>
## Part 4: 贝叶斯优化逆向设计

使用贝叶斯优化在VAE的潜在空间中搜索具有目标性质的MOF。

### 4.1 定义性质预测函数

In [ ]:
def predict_co2_uptake(structure_dict):
    """
    简化的CO2吸附量预测函数
    实际应用中，这里应该是一个训练好的性质预测模型
    """
    node_features = structure_dict['node_features']
    adj_matrix = structure_dict['adj_matrix']
    
    # 简化的启发式预测（仅用于演示）
    # 基于节点特征的均值和连接数
    mean_features = node_features.mean().item()
    num_edges = (adj_matrix > 0.5).sum().item()
    
    # 简单的线性组合
    predicted_uptake = 5.0 + 2.0 * mean_features + 0.01 * num_edges
    
    return predicted_uptake

# 测试预测函数
test_structure = interpolated_mofs[0]
test_prediction = predict_co2_uptake(test_structure)
print(f"测试预测: {test_prediction:.2f} mmol/g")

### 4.2 设置逆向设计器

In [ ]:
# 创建逆向设计器
latent_bounds = [[-3.0, 3.0]] * vae_model.latent_dim  # 潜在空间搜索范围

designer = MOFInverseDesigner(
    vae_model=vae_model,
    property_predictor=predict_co2_uptake,
    latent_bounds=latent_bounds,
    device=device
)

print(f"逆向设计器已创建")
print(f"潜在空间维度: {vae_model.latent_dim}")
print(f"搜索边界: [-3.0, 3.0] × {vae_model.latent_dim}")

### 4.3 运行贝叶斯优化

In [ ]:
# 优化目标：最大化CO2吸附量
print("开始贝叶斯优化...")
print("目标: 最大化CO2吸附量\n")

result = designer.optimize_for_property(
    target_property='CO2_uptake',
    n_iterations=20,  # 优化迭代次数
    acquisition='ei',  # 期望改进
    n_initial_points=5,  # 初始随机采样点
    verbose=True
)

print("\n优化完成！")
print(f"\n最优结果:")
print(f"  最高CO2吸附量: {result['best_property']:.2f} mmol/g")
print(f"  最优潜在向量: {result['best_latent'][:3]}... (显示前3维)")
print(f"  总评估次数: {len(result['optimization_history'][0])}")

### 4.4 可视化优化过程

In [ ]:
# 绘制优化历史
property_history = result['optimization_history'][1]
best_so_far = np.maximum.accumulate(property_history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 每次迭代的性质值
axes[0].plot(property_history, 'o-', label='Evaluated', alpha=0.6)
axes[0].plot(best_so_far, 'r-', linewidth=2, label='Best so far')
axes[0].axhline(y=result['best_property'], color='g', linestyle='--', 
                label=f'Optimum: {result["best_property"]:.2f}')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('CO2 Uptake (mmol/g)')
axes[0].set_title('优化历史')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 改进幅度
improvements = np.diff(best_so_far)
axes[1].bar(range(len(improvements)), improvements, alpha=0.7)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Improvement')
axes[1].set_title('每次迭代的改进')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n观察:")
print(f"  初始最大值: {property_history[:5].max():.2f} mmol/g")
print(f"  最终最优值: {result['best_property']:.2f} mmol/g")
print(f"  总改进: {result['best_property'] - property_history[:5].max():.2f} mmol/g")

### 4.5 解码最优MOF

In [ ]:
# 从最优潜在向量解码MOF结构
best_latent = torch.tensor(result['best_latent']).unsqueeze(0).to(device)

vae_model.eval()
with torch.no_grad():
    best_node, best_adj, best_edge = vae_model.decode(best_latent, num_nodes=30)

print("最优MOF结构:")
print(f"  节点特征: {best_node.shape}")
print(f"  邻接矩阵: {best_adj.shape}")
print(f"  边特征: {best_edge.shape}")
print(f"  边数: {(best_adj > 0.5).sum().item()}")
print(f"\n预测CO2吸附量: {result['best_property']:.2f} mmol/g")

---
<a id="part5"></a>
## Part 5: 可视化分析

深入分析生成的MOF和潜在空间结构。

### 5.1 潜在空间的性质分布

In [ ]:
# 在潜在空间的2D网格上评估性质
print("在潜在空间网格上评估CO2吸附量...")

# 使用PCA将潜在空间降到2D
pca = PCA(n_components=2)
latent_2d_pca = pca.fit_transform(latent_vectors)

print(f"PCA解释的方差: {pca.explained_variance_ratio_.sum():.2%}")

# 创建2D网格
x_min, x_max = latent_2d_pca[:, 0].min() - 1, latent_2d_pca[:, 0].max() + 1
y_min, y_max = latent_2d_pca[:, 1].min() - 1, latent_2d_pca[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 50),
    np.linspace(y_min, y_max, 50)
)

# 在网格点上评估（简化版本）
grid_points = np.c_[xx.ravel(), yy.ravel()]
grid_latent = pca.inverse_transform(grid_points)

grid_properties = []
for z_vec in grid_latent:
    z_tensor = torch.tensor(z_vec).unsqueeze(0).float().to(device)
    with torch.no_grad():
        node, adj, edge = vae_model.decode(z_tensor, num_nodes=30)
    prop = predict_co2_uptake({
        'node_features': node[0].cpu(),
        'adj_matrix': adj[0].cpu(),
        'edge_features': edge[0].cpu()
    })
    grid_properties.append(prop)

grid_properties = np.array(grid_properties).reshape(xx.shape)

# 绘制性质景观
plt.figure(figsize=(12, 9))
contour = plt.contourf(xx, yy, grid_properties, levels=20, cmap='viridis', alpha=0.8)
plt.colorbar(contour, label='Predicted CO2 Uptake (mmol/g)')

# 叠加训练数据点
plt.scatter(latent_2d_pca[:, 0], latent_2d_pca[:, 1], 
           c='red', s=20, alpha=0.5, edgecolors='white', linewidth=0.5,
           label='Training MOFs')

# 标记最优点
best_latent_2d = pca.transform(result['best_latent'].reshape(1, -1))
plt.scatter(best_latent_2d[0, 0], best_latent_2d[0, 1],
           c='yellow', s=300, marker='*', edgecolors='black', linewidth=2,
           label=f'Optimum ({result["best_property"]:.2f} mmol/g)', zorder=5)

plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('潜在空间中的CO2吸附量景观')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 5.2 比较原始MOF和生成MOF

In [ ]:
# 统计比较
print("原始训练集 vs 生成MOF的统计比较\n")

# 收集生成MOF的特征
num_gen_samples = 50
gen_node_means = []
gen_edge_counts = []

with torch.no_grad():
    for _ in range(num_gen_samples):
        node, adj, edge = vae_model.sample(num_samples=1)
        gen_node_means.append(node[0].mean().item())
        gen_edge_counts.append((adj[0] > 0.5).sum().item())

# 收集原始MOF的特征
orig_node_means = [data.x.mean().item() for data in train_dataset[:num_gen_samples]]
orig_edge_counts = [data.edge_index.shape[1] for data in train_dataset[:num_gen_samples]]

# 绘制对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 节点特征均值分布
axes[0].hist(orig_node_means, bins=15, alpha=0.6, label='Original', color='blue')
axes[0].hist(gen_node_means, bins=15, alpha=0.6, label='Generated', color='orange')
axes[0].set_xlabel('Mean Node Feature Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('节点特征均值分布')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 边数分布
axes[1].hist(orig_edge_counts, bins=15, alpha=0.6, label='Original', color='blue')
axes[1].hist(gen_edge_counts, bins=15, alpha=0.6, label='Generated', color='orange')
axes[1].set_xlabel('Number of Edges')
axes[1].set_ylabel('Frequency')
axes[1].set_title('边数分布')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察: 生成的MOF应该与原始MOF具有相似的统计特性")

---
<a id="part6"></a>
## Part 6: 端到端逆向设计流程

完整的工作流程：从目标性质到生成满足要求的MOF。

In [ ]:
class CompleteMOFDesignPipeline:
    """
    完整的MOF设计流程
    """
    def __init__(self, vae_model, property_predictor, device='cpu'):
        self.vae_model = vae_model
        self.property_predictor = property_predictor
        self.device = device
        
    def design_mof(self, target_property, target_value, constraints=None, verbose=True):
        """
        设计满足目标性质的MOF
        
        参数:
            target_property: 目标性质名称
            target_value: 目标值
            constraints: 额外约束（可选）
        """
        if verbose:
            print(f"="*60)
            print(f"MOF逆向设计流程")
            print(f"="*60)
            print(f"目标: {target_property} ≥ {target_value}")
            if constraints:
                print(f"约束: {constraints}")
            print()
        
        # Step 1: 贝叶斯优化
        if verbose:
            print("Step 1: 在潜在空间中搜索...")
        
        latent_bounds = [[-3.0, 3.0]] * self.vae_model.latent_dim
        designer = MOFInverseDesigner(
            vae_model=self.vae_model,
            property_predictor=self.property_predictor,
            latent_bounds=latent_bounds,
            device=self.device
        )
        
        result = designer.optimize_for_property(
            target_property=target_property,
            n_iterations=30,
            acquisition='ei',
            verbose=False
        )
        
        if verbose:
            print(f"  找到最优点: {result['best_property']:.2f}")
        
        # Step 2: 解码生成MOF
        if verbose:
            print("\nStep 2: 从潜在空间解码MOF结构...")
        
        best_latent = torch.tensor(result['best_latent']).unsqueeze(0).to(self.device)
        self.vae_model.eval()
        with torch.no_grad():
            node, adj, edge = self.vae_model.decode(best_latent, num_nodes=30)
        
        mof_structure = {
            'node_features': node[0].cpu(),
            'adj_matrix': adj[0].cpu(),
            'edge_features': edge[0].cpu(),
            'latent_vector': result['best_latent']
        }
        
        if verbose:
            print(f"  生成的MOF: {node[0].shape[0]} 个原子, "
                  f"{(adj[0] > 0.5).sum().item()} 条键")
        
        # Step 3: 验证
        if verbose:
            print("\nStep 3: 验证性质...")
        
        predicted = self.property_predictor(mof_structure)
        
        if verbose:
            print(f"  预测{target_property}: {predicted:.2f}")
            if predicted >= target_value:
                print(f"  ✓ 满足目标 ({target_value})")
            else:
                print(f"  ✗ 未达到目标 ({target_value})")
        
        # Step 4: 返回结果
        return {
            'structure': mof_structure,
            'predicted_property': predicted,
            'target_met': predicted >= target_value,
            'optimization_result': result
        }

# 创建流程
pipeline = CompleteMOFDesignPipeline(
    vae_model=vae_model,
    property_predictor=predict_co2_uptake,
    device=device
)

print("完整设计流程已创建！")

In [ ]:
# 示例1: 设计高CO2吸附量的MOF
result1 = pipeline.design_mof(
    target_property='CO2_uptake',
    target_value=7.0,
    verbose=True
)

print(f"\n{'='*60}")
print(f"设计完成！")
print(f"{'='*60}")

In [ ]:
# 示例2: 设计多个满足不同目标的MOF
targets = [6.0, 7.0, 8.0]
print("\n设计多个MOF，满足不同的CO2吸附量目标...\n")

designed_mofs = []
for i, target in enumerate(targets, 1):
    print(f"设计 #{i}: 目标 = {target} mmol/g")
    result = pipeline.design_mof(
        target_property='CO2_uptake',
        target_value=target,
        verbose=False
    )
    designed_mofs.append(result)
    print(f"  结果: {result['predicted_property']:.2f} mmol/g"
          f" {'✓' if result['target_met'] else '✗'}\n")

print(f"成功设计了 {len(designed_mofs)} 个MOF！")

---
<a id="part7"></a>
## Part 7: 总结与练习

### 学习要点总结

1. **VAE（变分自编码器）**
   - ELBO损失 = 重构损失 + β × KL散度
   - 学习MOF的连续潜在表示
   - 支持插值和采样生成

2. **扩散模型**
   - 通过逐步去噪生成高质量结构
   - DDPM算法：前向加噪 + 反向去噪
   - 噪声调度策略影响生成质量

3. **贝叶斯优化**
   - 高斯过程建立代理模型
   - 采集函数平衡探索与利用
   - 在潜在空间中搜索目标性质

4. **逆向设计流程**
   - 目标设定 → 优化搜索 → 结构解码 → 验证
   - 可以设计满足特定性质的新MOF

### 练习题

#### 练习1: 调整VAE超参数
尝试不同的超参数组合，观察对潜在空间的影响：
- 改变`latent_dim`（8, 16, 32）
- 改变`beta`值（0.1, 0.5, 1.0）
- 观察生成MOF的质量

#### 练习2: 多目标优化
修改优化目标，同时优化：
- 高CO2吸附量
- 适中的结构复杂度（边数）
- 提示：使用加权和或Pareto优化

#### 练习3: 比较不同采集函数
在贝叶斯优化中比较三种采集函数的效果：
- Expected Improvement (EI)
- Upper Confidence Bound (UCB)
- Probability of Improvement (PI)

#### 练习4: 条件生成
扩展VAE为条件VAE（cVAE）：
- 在编码器和解码器中引入条件信息（如目标性质）
- 直接生成满足条件的MOF

#### 练习5: 真实数据测试
在真实MOF数据集上测试：
- 下载QMOF或CoRE-MOF数据
- 训练VAE和性质预测模型
- 执行逆向设计

### 扩展阅读

1. **VAE相关**
   - Kingma & Welling (2014): "Auto-Encoding Variational Bayes"
   - Kusner et al. (2017): "Grammar VAE" for molecules

2. **扩散模型**
   - Ho et al. (2020): "Denoising Diffusion Probabilistic Models"
   - Hoogeboom et al. (2022): "E(3) Equivariant Diffusion"

3. **贝叶斯优化**
   - Shahriari et al. (2016): "Taking the Human Out of the Loop: A Review of Bayesian Optimization"
   - Griffiths & Hernández-Lobato (2020): "Constrained Bayesian Optimization"

4. **MOF逆向设计**
   - Yao et al. (2021): "Inverse Design of Nanoporous Crystalline Reticular Materials"
   - Moosavi et al. (2020): "Understanding the Diversity of the MOF Ecosystem"

### 下一步

- **Day 5**: 大语言模型（LLM）在MOF智能设计中的应用
  - Text-to-Structure生成
  - 文献挖掘与知识抽取
  - RAG检索增强生成
  - 自主发现智能体

继续学习 → `notebooks/day5_tutorial.ipynb`

---
## 恭喜完成Day 4教程！ 🎉

你已经掌握了：
- ✅ VAE训练和潜在空间分析
- ✅ 扩散模型生成
- ✅ 贝叶斯优化逆向设计
- ✅ 端到端MOF设计流程

继续探索生成模型在材料科学中的无限可能！